In [ ]:
import glob
import random
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


In [ ]:
TARGET_COL = "Jc"
FEATURE_COLS = ["T", "B", "theta"]

DATA_GLOB = "data/*.csv"
OUTPUT_CSV = "mlp_predictions.csv"
XGB_OUTPUT_CSV = "xgb_predictions.csv"
RELATIVE_ERROR_CSV = "mlp_relative_errors.csv"
PARITY_PLOT_PATH = "parity_plot.png"

TEST_SIZE = 0.2
RANDOM_STATE = 42
BATCH_SIZE = 64
EPOCHS = 200
LR = 1e-3
HIDDEN_DIMS = [64, 64]

USE_LOG_TARGET = True
USE_TRIG_FEATURES = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int]):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if USE_TRIG_FEATURES and "theta" in df.columns:
        df["sin_theta"] = np.sin(df["theta"])
        df["cos_theta"] = np.cos(df["theta"])

    return df

In [ ]:
def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_data(data_glob: str = DATA_GLOB):
    files = sorted(glob.glob(data_glob))
    if not files:
        raise FileNotFoundError(
            f"No CSV files found for pattern: {data_glob}. "
            f"Current working directory: {Path.cwd()}"
        )

    df_list = [pd.read_csv(file_path) for file_path in files]
    df = pd.concat(df_list, ignore_index=True)
    df = df.rename(
        columns={
            "Temperature (K)": "T",
            "Applied field (T)": "B",
            "Applied field angle (°)": "theta",
            "Critical current (A)": "Jc",
        }
    )

    required_cols = FEATURE_COLS + [TARGET_COL]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    df = df.dropna(subset=required_cols)
    df = build_features(df)

    feature_cols = FEATURE_COLS.copy()
    if USE_TRIG_FEATURES and "theta" in FEATURE_COLS:
        feature_cols = [col for col in feature_cols if col != "theta"]
        feature_cols += ["sin_theta", "cos_theta"]

    X = df[feature_cols].to_numpy()
    y = df[TARGET_COL].to_numpy()

    if USE_LOG_TARGET:
        if np.any(y <= 0):
            raise ValueError("Jc must be positive when USE_LOG_TARGET=True.")
        y = np.log(y)

    return X, y, feature_cols


def evaluate(model, loader, device):
    model.eval()
    preds = []
    targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            output = model(X_batch)
            preds.append(output.cpu().numpy())
            targets.append(y_batch.cpu().numpy())

    preds = np.vstack(preds).ravel()
    targets = np.vstack(targets).ravel()

    return preds, targets


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    return running_loss / len(loader.dataset)


def save_parity_plot(pred_csv_path: str = OUTPUT_CSV, output_path: str = PARITY_PLOT_PATH) -> None:
    pred_df = pd.read_csv(pred_csv_path)

    y_true = pred_df["y_true"]
    y_pred = pred_df["y_pred"]

    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, s=12, alpha=0.7)
    plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5)
    plt.xlabel("True Jc")
    plt.ylabel("Predicted Jc")
    plt.title("Parity Plot (MLP)")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

    print(f"Saved: {output_path}")


def check_xgboost_dependency() -> None:
    if importlib.util.find_spec("xgboost") is None:
        raise ModuleNotFoundError(
            "Missing dependency: xgboost. Install with `pip install xgboost`."
        )


In [ ]:
def main():
    set_seed(RANDOM_STATE)

    X, y, used_feature_cols = load_data()
    print("Features:", used_feature_cols)
    print("Shape:", X.shape, y.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_dataset = TabularDataset(X_train, y_train)
    test_dataset = TabularDataset(X_test, y_test)

    use_pin_memory = DEVICE == "cuda"
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=use_pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=use_pin_memory,
    )

    model = MLPRegressor(input_dim=X_train.shape[1], hidden_dims=HIDDEN_DIMS).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        epoch_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:03d} | Train Loss: {epoch_loss:.6f}")

    pred_log, true_log = evaluate(model, test_loader, DEVICE)

    if USE_LOG_TARGET:
        mlp_pred = np.exp(pred_log)
        mlp_true = np.exp(true_log)
    else:
        mlp_pred = pred_log
        mlp_true = true_log

    mlp_r2 = r2_score(mlp_true, mlp_pred)
    mlp_rmse = np.sqrt(mean_squared_error(mlp_true, mlp_pred))

    print(f"MLP Test R2:   {mlp_r2:.6f}")
    print(f"MLP Test RMSE: {mlp_rmse:.6f}")

    result_df = pd.DataFrame({"y_true": mlp_true, "y_pred": mlp_pred})
    result_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV}")

    result_df["relative_error"] = np.abs(result_df["y_pred"] - result_df["y_true"]) / result_df["y_true"]
    mean_relative_error = result_df["relative_error"].mean()
    result_df[["y_true", "y_pred", "relative_error"]].to_csv(RELATIVE_ERROR_CSV, index=False)
    print(f"MLP Mean Relative Error: {mean_relative_error:.6f}")
    print(f"Saved: {RELATIVE_ERROR_CSV}")

    save_parity_plot(OUTPUT_CSV, PARITY_PLOT_PATH)

    check_xgboost_dependency()
    from xgboost import XGBRegressor

    xgb_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_model.fit(X_train, y_train)

    xgb_pred_log = xgb_model.predict(X_test)
    if USE_LOG_TARGET:
        xgb_pred = np.exp(xgb_pred_log)
        xgb_true = np.exp(y_test)
    else:
        xgb_pred = xgb_pred_log
        xgb_true = y_test

    xgb_r2 = r2_score(xgb_true, xgb_pred)
    xgb_rmse = np.sqrt(mean_squared_error(xgb_true, xgb_pred))

    xgb_df = pd.DataFrame({"y_true": xgb_true, "y_pred": xgb_pred})
    xgb_df.to_csv(XGB_OUTPUT_CSV, index=False)
    print(f"Saved: {XGB_OUTPUT_CSV}")

    print("\nModel Comparison on Current Split")
    print(f"MLP      -> R2: {mlp_r2:.6f}, RMSE: {mlp_rmse:.6f}")
    print(f"XGBoost  -> R2: {xgb_r2:.6f}, RMSE: {xgb_rmse:.6f}")

    if xgb_r2 > mlp_r2:
        better_model = "XGBoost"
    elif xgb_r2 < mlp_r2:
        better_model = "MLP"
    else:
        better_model = "Tie"

    print(f"Better model on this split: {better_model}")


if __name__ == "__main__":
    main()


In [ ]:
# Optional quick check for saved outputs
pd.read_csv(OUTPUT_CSV).head()


In [ ]:
pd.read_csv(XGB_OUTPUT_CSV).head()
